# Notebook 02 — Event Definition & Detection
**Thesis:** Chapter 6 | **Input:** `data/reconstructed.csv` | **Output:** `data/events.csv`

---

## Purpose
Adds 18 event indicator columns to the reconstructed dataset.

An **event** is a binary or categorical condition computed from the observable variables  
at the time of each received packet. For example: "Was CO₂ above 700 ppm at this moment?"

Events are organised into six families:

| Family | Events | Scientific motivation |
|--------|--------|----------------------|
| Occupancy (CO₂) | E1, E2, E3 | CO₂ is the best non-invasive proxy for indoor human presence |
| Temporal | E4, E5, E6 | Office activity follows predictable daily and weekly cycles |
| Air Quality (PM2.5) | E7, E8 | PM2.5 spikes mark activity bursts (cleaning, printing, cooking) |
| Atmospheric | E9, E10, E11, E12 | Pressure, humidity, temperature vary seasonally and with HVAC |
| Transmission Config | E13, E14, E15 | SF directly controls Time-on-Air and collision probability |
| Burst Loss | E16 | Classifies loss episodes by severity |
| Signal Context | E17, E18 | RSSI/ESP of packet before a loss — proxy for channel state |

**All events are added to the full dataset.**  
Event-conditioned analysis in notebook 03 then filters to `PDR_link` intervals  
(excluding outages and SF artifacts) before computing correlations.

## 0 · Imports & Load

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'reconstructed.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values(['device_id', 'time']).reset_index(drop=True)

print(f'Rows    : {len(df):,}')
print(f'Devices : {sorted(df["device_id"].unique())}')
print(f'Columns : {df.columns.tolist()}')

Rows    : 1,217,313
Devices : ['ED0', 'ED1', 'ED2', 'ED3', 'ED4', 'ED5']
Columns : ['device_id', 'time', 'co2', 'humidity', 'pm25', 'pressure', 'temperature', 'rssi', 'snr', 'esp', 'SF', 'frequency', 'toa', 'distance', 'c_walls', 'w_walls', 'exp_pl', 'n_power', 'f_count', 'p_count', 'mac_to_radio_loss', 'app_to_mac_drop', 'total_loss', 'total_tx', 'is_reset', 'is_outage', 'is_sf_artifact', 'loss_zone']


## 1 · Temporal Events  *(§6.2)*

Time is converted to local German time (`Europe/Berlin`) before computing hour, weekday, and season.  
This handles both DST transitions in the campaign automatically — UTC alone would misplace events  
around the October 2024 and March 2025 clock changes.

Four events in this family:
- **E2** — weekday vs weekend: the coarsest occupancy signal
- **E4** — time-of-day band: captures the office diurnal cycle
- **E5** — office hours: binary version of E4 for simple comparisons
- **E6** — season: captures the autumn/winter/spring variation over 34 weeks

In [2]:
# convert to local time — avoids DST errors on hour/weekday extraction
local = df['time'].dt.tz_convert('Europe/Berlin')
df['hour']      = local.dt.hour
df['dayofweek'] = local.dt.dayofweek   # 0=Monday, 6=Sunday
df['month']     = local.dt.month

# E2: weekday (1) vs weekend (0)
# Campaign has ~71% weekday rows — consistent with a 5-day working week
df['e2_is_weekday'] = (df['dayofweek'] < 5).astype(int)

# E4: four time-of-day bands matching the office activity profile
# CO2 data confirms: rises from 08:00, peaks 11:00-17:00, drops after 18:00
df['e4_time_of_day'] = pd.cut(
    df['hour'],
    bins=[-2, 4, 10, 17, 22],
    labels=['night', 'morning', 'peak', 'evening']
)

# E5: office hours = weekday AND between 08:00 and 17:59
# Binary aggregation of E4 for clean event conditioning
df['e5_office_hours'] = (
    (df['e2_is_weekday'] == 1) & df['hour'].between(8, 17)
).astype(int)

# E6: season — the 34-week campaign covers autumn 2024, winter 2024/25, spring 2025
df['e6_season'] = df['month'].map({
    9: 'autumn', 10: 'autumn', 11: 'autumn',
    12: 'winter',  1: 'winter',  2: 'winter',
     3: 'spring',  4: 'spring',  5: 'spring'
})

# quick verification
print(f'Weekday share : {df["e2_is_weekday"].mean()*100:.1f}%')
print(f'Office hours  : {df["e5_office_hours"].mean()*100:.1f}%')
print(f'Season counts : {df["e6_season"].value_counts().to_dict()}')

Weekday share : 70.9%
Office hours  : 29.2%
Season counts : {'spring': 443050, 'winter': 428825, 'autumn': 345438}


## 2 · Occupancy Events — CO₂  *(§6.1)*

CO₂ is the primary occupancy proxy. In an unventilated space, CO₂ rises linearly  
with the number of people present (human metabolism produces ~200 ml CO₂/min at rest).

Outdoor ambient CO₂ is ~420 ppm. In this office building:
- Weekday mean: **577 ppm** (std 139) — people at desks
- Weekend mean: **454 ppm** (std 47) — building largely empty

Three events:
- **E1** — CO₂ tier: three-level occupancy proxy (background / moderate / high)
- **E3** — CO₂ rising: rapid increase signals people arriving

In [6]:
# E1: CO2 concentration tier — three occupancy levels
# Thresholds: 500 ppm = outdoor ambient ceiling, 700 ppm = moderate occupancy onset
df['e1_co2_tier'] = pd.cut(
    df['co2'],
    bins=[0, 500, 700, 10_000],
    labels=['background', 'moderate', 'high']
)

# E3: CO2 rising — first difference > 20 ppm per 60-second interval
# Captures the transition moment when occupancy is actively increasing
# Computed per device to avoid cross-device artifacts
df['co2_delta']     = df.groupby('device_id')['co2'].diff()
df['e3_co2_rising'] = (df['co2_delta'] > 20).astype(int)

print('CO2 tier distribution:')
print(df['e1_co2_tier'].value_counts(normalize=True).mul(100).round(1).astype(str).add('%').to_string())
print(f'\nWeekday CO2 mean : {df[df["e2_is_weekday"]==1]["co2"].mean():.0f} ppm')
print(f'Weekend CO2 mean : {df[df["e2_is_weekday"]==0]["co2"].mean():.0f} ppm')

CO2 tier distribution:
e1_co2_tier
background    51.7%
moderate      36.0%
high          12.3%

Weekday CO2 mean : 576 ppm
Weekend CO2 mean : 457 ppm


## 3 · Air Quality Events — PM2.5  *(§6.3)*

PM2.5 (particulate matter < 2.5 µm) reflects airborne particles from cleaning,  
printing, cooking in nearby spaces, or external air quality events.

PM2.5 does not directly attenuate 868 MHz radio signals, but it correlates strongly  
with human activity bursts that temporarily change the propagation environment  
(furniture movement, increased foot traffic, door openings).

Two events:
- **E7** — device-specific spike: above each device's own 90th percentile  
  (accounts for the fact that ED5 is near a kitchen and has a much higher baseline than ED4)
- **E8** — absolute WHO tier: globally comparable classification

In [7]:
# E7: PM2.5 spike — device-specific 90th percentile threshold
# Using device-specific thresholds avoids penalising devices in dustier locations
# transform() applies the quantile per device and broadcasts back to row level
pm25_q90 = df.groupby('device_id')['pm25'].transform(lambda x: x.quantile(0.90))
df['e7_pm25_spike'] = (df['pm25'] > pm25_q90).astype(int)

# E8: PM2.5 absolute tier — WHO indoor air quality thresholds
df['e8_pm25_tier'] = pd.cut(
    df['pm25'],
    bins=[-0.01, 2, 10, 10_000],
    labels=['clean', 'moderate', 'elevated']
)

print('Device-specific PM2.5 90th percentile thresholds:')
print(df.groupby('device_id')['pm25'].quantile(0.90).round(2).to_string())
print(f'\nSpike rate (E7=1): {df["e7_pm25_spike"].mean()*100:.1f}%')

Device-specific PM2.5 90th percentile thresholds:
device_id
ED0    3.86
ED1    4.02
ED2    3.46
ED3    4.05
ED4    0.79
ED5    6.28

Spike rate (E7=1): 10.0%


## 4 · Atmospheric Events  *(§6.4)*

Four events covering pressure, humidity, and temperature.

**Pressure (E9, E10):** Barometric pressure in this indoor environment reflects  
seasonal weather patterns and HVAC operation. Rapid pressure drops (E10) correspond  
to large doors opening between pressurised corridors and stairwells, or HVAC mode switches.  
The campaign spans 287–347 hPa across autumn, winter, and spring.

**Humidity (E11):** Relative humidity ranges from ~25% (winter heating) to ~75% (autumn).  
High humidity weakly attenuates 868 MHz but more importantly correlates with  
weather conditions that affect building occupancy and HVAC behaviour.

**Temperature (E12):** Indoor temperature is HVAC-controlled. Quartile tiers capture  
seasonal thermal state without being affected by HVAC set-point changes between seasons.

In [11]:
# E9: pressure tier — campaign quartiles capture seasonal pressure variation
# Absolute thresholds avoided because altitude correction shifts baseline across seasons
df['e9_pressure_tier'] = pd.qcut(
    df['pressure'], q=4,
    labels=['low', 'medium_low', 'medium_high', 'high'],
    duplicates='drop'
)

# E10: pressure drop — first difference < -0.5 hPa per 60-second interval
# Captures HVAC switching or door-opening pressure transients
# todo 300 hpa and siegen is app. 970 hpaBut with weirdly low values? At sea level, normal pressure is ~1013 hPa. At the top of Mount Everest (8849m), pressure is ~314 hPa. Siegen at 300m would be ~978 hPa. So 287–348 hPa would place you higher than Mount Everest if taken literally. That is physically impossible indoors.

df['pressure_delta']    = df.groupby('device_id')['pressure'].diff()
df['e10_pressure_drop'] = (df['pressure_delta'] < -0.5).astype(int)

# E11: humidity tier — absolute bands aligned with comfort and RF propagation ranges
df['e11_humidity_tier'] = pd.cut(
    df['humidity'],
    bins=[0, 40, 55, 70, 101],
    labels=['dry', 'normal', 'humid', 'very_humid']
)

# E12: temperature tier — quartile-based because HVAC set-point shifts seasonally
df['e12_temp_tier'] = pd.qcut(
    df['temperature'], q=4,
    labels=['cold', 'cool', 'warm', 'hot'],
    duplicates='drop'
)

print(f'Pressure range  : {df["pressure"].min():.1f} – {df["pressure"].max():.1f} hPa')
print(f'Humidity range  : {df["humidity"].min():.1f} – {df["humidity"].max():.1f} %')
print(f'Temperature range: {df["temperature"].min():.1f} – {df["temperature"].max():.1f} °C')
print(f'Pressure drops  : {df["e10_pressure_drop"].sum():,} intervals')

Pressure range  : 286.9 – 347.6 hPa
Humidity range  : 14.0 – 60.2 %
Temperature range: 13.7 – 44.0 °C
Pressure drops  : 160 intervals


## 5 · Transmission Configuration Events  *(§6.5)*

Spreading Factor (SF) directly controls Time-on-Air (ToA) — the duration a packet  
occupies the radio channel. Under LoRaWAN's ALOHA protocol (no collision avoidance),  
longer ToA means a wider window during which another device's transmission can collide.

This creates a fundamental tension:
- Higher SF → better range and sensitivity
- Higher SF → longer ToA → higher collision probability

In this deployment, ADR was disabled and SFs were manually rotated.  
The SF × occupancy interaction (E14 × E1) is the novel joint finding of this thesis:  
does higher SF suffer disproportionately during busy periods?

| SF | ToA (ms) | Collision window vs SF7 |
|----|---------|------------------------|
| 7 | 71.9 | baseline |
| 8 | 133.6 | 1.9× |
| 9 | 246.8 | 3.4× |
| 10 | 452.6 | 6.3× |

In [12]:
TOA_MS = {7: 71.9, 8: 133.6, 9: 246.8, 10: 452.6}   # ms at BW=125kHz, 26-byte payload

# E13: spreading factor — categorical (4 values: 7, 8, 9, 10)
df['e13_sf'] = df['SF'].astype(int)

# E14: SF tier — binary split on ToA boundary
# low_sf (7-8): ToA ≤ 134ms | high_sf (9-10): ToA ≥ 247ms
df['e14_sf_tier'] = df['e13_sf'].apply(
    lambda s: 'low_sf' if s in (7, 8) else 'high_sf'
)

# E15: Time-on-Air class — derived from SF, expresses collision risk explicitly
df['e15_toa_ms']    = df['e13_sf'].map(TOA_MS)
df['e15_toa_class'] = df['e14_sf_tier'].map({'low_sf': 'short_toa', 'high_sf': 'long_toa'})

print('SF distribution:')
print(df['e13_sf'].value_counts(normalize=True).mul(100).sort_index().round(1).to_string())

SF distribution:
e13_sf
7     26.0
8     25.8
9     24.7
10    23.4


## 6 · Burst Loss Event  *(§6.6)*

A burst is a sequence of consecutively lost packets. Bursts are qualitatively different  
from isolated losses because they create **contiguous data gaps** that cannot be filled  
by interpolation — critical for environmental monitoring applications.

**Threshold B=3:** devices transmit every 60 seconds, so 3 consecutive losses = 3 minutes of missing data.  
Three minutes is the minimum gap that visibly disrupts time-series analyses of slowly-varying  
environmental signals like CO₂ and temperature.

Four loss types:
- `no_loss` — packet arrived right after previous one (no gap)
- `isolated` — exactly 1 packet lost (single random collision)
- `small_burst` — 2–4 consecutive losses (brief channel degradation)
- `large_burst` — 5+ consecutive losses (sustained outage)

> Note: E16 uses `mac_to_radio_loss` — the radio-layer counter.  
> SF artifacts and infrastructure outages are already flagged separately  
> and excluded from event-conditioned analysis in notebook 03.

In [15]:
# E16: loss type classification based on consecutive radio losses per interval
df['e16_loss_type'] = pd.cut(
    df['mac_to_radio_loss'].where(~df['is_sf_artifact'], 0),
    bins=[-1, 0, 1, 4, 10_000_000],
    labels=['no_loss', 'isolated', 'small_burst', 'large_burst']
)

print('Loss type distribution (all intervals):')
dist = df['e16_loss_type'].value_counts(normalize=True).mul(100).round(1)
print(dist.astype(str).add('%').to_string())

Loss type distribution (all intervals):
e16_loss_type
no_loss        97.8%
isolated        1.2%
large_burst     0.6%
small_burst     0.4%


In [16]:
# check what loss values are behind each category
print("Loss value counts by e16 category:")
for zone in ['isolated', 'small_burst', 'large_burst']:
    mask = df['e16_loss_type'] == zone
    print(f"\n{zone} ({mask.sum():,} intervals):")
    print(df[mask]['mac_to_radio_loss'].value_counts().sort_index().head(10).to_string())

Loss value counts by e16 category:

isolated (15,024 intervals):
mac_to_radio_loss
1    15024

small_burst (4,450 intervals):
mac_to_radio_loss
2    3125
3    1050
4     275

large_burst (6,727 intervals):
mac_to_radio_loss
5      199
6      762
7     2383
8      254
9     2528
10     186
11      56
12      23
13      25
14      16


In [17]:
mask = df['e16_loss_type'] == 'large_burst'
print(df[mask].groupby(['mac_to_radio_loss', 'SF']).size().sort_values(ascending=False).head(15))

mac_to_radio_loss  SF
9                  10    2482
7                  10    2351
6                  10     701
8                  10     219
10                 10     169
5                  10     119
11                 10      48
6                  8       39
5                  8       38
                   7       36
8                  8       26
7                  8       22
9                  7       20
                   8       20
12                 10      16
dtype: int64


## 7 · Signal Context Events  *(§6.7)*

RSSI and ESP are only available for **received** packets — never for lost ones.  
To characterise the channel state around a loss episode, we use the signal quality  
of the packet received **immediately before** the loss interval.

This is a one-sided look-back — the signal quality just before the gap.  
A future refinement could average both the pre-loss and post-loss packets  
to better bracket the channel state (proposed in §8.6 as a methodological extension).

Two events:
- **E17** — RSSI tier: strong ≥ −70 dBm / moderate −90 to −70 / weak < −90
- **E18** — ESP tier: effective signal power combining RSSI and SNR (see §2.2.3)

In [14]:
# E17: pre-loss RSSI tier — signal strength of the packet before each interval
# Thresholds reflect the receiver sensitivity range for SF7-SF10 (Table 2.2)
df['e17_rssi_tier'] = pd.cut(
    df['rssi'],
    bins=[-200, -90, -70, 0],
    labels=['weak', 'moderate', 'strong']
)

# E18: ESP tier — campaign quartile-based (ESP = RSSI + SNR adjustment, see §2.2.3)
# Quartile tiers used because ESP has a continuous distribution without obvious breakpoints
df['e18_esp_tier'] = pd.qcut(
    df['esp'], q=3,
    labels=['low_esp', 'medium_esp', 'high_esp'],
    duplicates='drop'
)

print('RSSI tier distribution:')
print(df['e17_rssi_tier'].value_counts(normalize=True).mul(100).round(1).to_string())
print(f'\nRSSI range: {df["rssi"].min():.0f} to {df["rssi"].max():.0f} dBm')
print(f'ESP  range: {df["esp"].min():.1f} to {df["esp"].max():.1f} dBm')

RSSI tier distribution:
e17_rssi_tier
strong      51.0
moderate    28.9
weak        20.1

RSSI range: -128 to -29 dBm
ESP  range: -141.1 to -29.8 dBm


## 8 · Event Summary  *(Table 6.1)*

Verify all 18 events are present and print the complete taxonomy.

In [18]:
event_cols = [c for c in df.columns if c.startswith('e') and '_' in c]

print(f'Total event columns: {len(event_cols)}')
print()
# todo ADD THE FAMILY CATEGORIES TOO TO MAKE IT MORE INTERESTING, AND MATCH TABLE 6.1...
# print taxonomy with coverage statistics
print(f"{'Event':<25} {'Type':<12} {'Coverage / Values'}")
print('-' * 70)
for col in sorted(event_cols):
    dtype = df[col].dtype
    if str(dtype) in ['int64', 'float64', 'bool']:
        coverage = f'{df[col].mean()*100:.1f}% active'
    else:
        coverage = str(df[col].unique().tolist())
    print(f'{col:<25} {str(dtype):<12} {coverage}')

Total event columns: 20

Event                     Type         Coverage / Values
----------------------------------------------------------------------
e10_pressure_drop         int64        0.0% active
e11_humidity_tier         category     ['normal', 'dry', 'humid']
e12_temp_tier             category     ['hot', 'warm', 'cool', 'cold']
e13_sf                    int64        845.4% active
e14_sf_tier               str          ['high_sf', 'low_sf']
e15_toa_class             str          ['long_toa', 'short_toa']
e15_toa_ms                float64      22011.1% active
e16_loss_type             category     ['no_loss', 'isolated', 'large_burst', 'small_burst']
e17_rssi_tier             category     ['strong', 'weak', 'moderate']
e18_esp_tier              category     ['high_esp', 'low_esp', 'medium_esp']
e1_co2_tier               category     ['moderate', 'background', 'high']
e2_is_weekday             int64        70.9% active
e3_co2_rising             int64        0.3% active
e4_time_

## 9 · Save

In [ ]:
df.to_csv(DATA_DIR / 'events.csv', index=False)

event_cols = [c for c in df.columns if c.startswith('e') and '_' in c]
print(f'Saved  : {DATA_DIR / "events.csv"}')
print(f'Rows   : {len(df):,}')
print(f'Columns: {len(df.columns)} total | {len(event_cols)} event columns')